In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import glob
from pathlib import Path
import time

I0000 00:00:1779312981.632097  408704 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
PROJECT_ROOT = r"/home/hasan/coding/MoneyLens/ai"
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.ocr_config import *
from src.ocr_model import build_ocr_model, CTCLayer
from src.text_encoder import encode_text, decode_prediction

I0000 00:00:1779313020.454826  408704 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1765 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [3]:
BASE_DIR      = r"/home/hasan/coding/MoneyLens/ai/Dataset_ocr"
PREP_DIR      = os.path.join(BASE_DIR, "preprocessed")
MODEL_DIR     = os.path.join(PROJECT_ROOT, "saved_model")
LOG_DIR       = os.path.join(PROJECT_ROOT, "tensorboard_logs")
CHECKPOINT_DIR= os.path.join(MODEL_DIR, "checkpoints")
 
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
 
BATCH_SIZE    = 32
EPOCHS        = 100
LEARNING_RATE = 1e-3
VALIDATION_SPLIT = 0.1
PATIENCE      = 10

In [4]:
def load_dataset_from_npy_and_csv(split: str):
    """
    Load gambar dari .npy + label dari _annotations.csv
    
    Returns:
      images: np.ndarray (N, 32, 128, 1)
      labels: np.ndarray (N, MAX_TEXT_LENGTH)
      classes: list of str (nama kelas per sampel)
    """
    arrays_dir = os.path.join(PREP_DIR, split, "arrays")
    csv_path   = os.path.join(BASE_DIR, split, "_annotations.csv")
 
    if not os.path.exists(arrays_dir):
        print(f"  [ERROR] Folder tidak ada: {arrays_dir}")
        return None, None, None
 
    if not os.path.exists(csv_path):
        print(f"  [ERROR] CSV tidak ada: {csv_path}")
        return None, None, None
 
    # Baca annotations
    df = pd.read_csv(csv_path)
    print(f"  {split}: {len(df)} anotasi | "
          f"kelas: {sorted(df['class'].unique().tolist())}")
 
    # Buat mapping filename → class untuk quick lookup
    filename_to_class = {}
    for _, row in df.iterrows():
        fname = row["filename"]
        cls   = row["class"]
        if fname not in filename_to_class:
            filename_to_class[fname] = []
        filename_to_class[fname].append(cls)
 
    images, labels, classes = [], [], []
 
    # Baca .npy files
    npy_files = sorted(glob.glob(os.path.join(arrays_dir, "*.npy")))
    print(f"  {len(npy_files)} file .npy ditemukan")
 
    for npy_path in npy_files:
        # Ekstrak filename asli dari nama file .npy
        # Format: X123_class_00.npy → X123_... (gambar original)
        npy_fname = Path(npy_path).stem  # contoh: X123_harga_satuan_00
        
        # Cari filename original (substring match)
        original_fname = None
        for fname in filename_to_class.keys():
            if fname.split(".")[0] in npy_fname:  # cocokkan tanpa extension
                original_fname = fname
                break
 
        if original_fname is None:
            continue
 
        # Ambil kelas dari nama file .npy
        cls = None
        for label_cls in CLASS_LABELS:
            if label_cls in npy_fname:
                cls = label_cls
                break
 
        if cls is None:
            continue
 
        try:
            arr = np.load(npy_path, allow_pickle=False)
            
            # Validate shape
            if arr.shape != (IMG_H, IMG_W, CHANNELS):
                continue
 
            # Ground truth: gunakan nama kelas sebagai teks
            # (pada proyek nyata: ganti dengan teks OCR yang sebenarnya)
            text = cls.lower()
            lbl  = encode_text(text).numpy()
 
            # ✅ PERBAIKAN: Simpan actual label length SEBELUM padding
            actual_len = len(lbl)
 
            # Pad ke MAX_TEXT_LENGTH
            if len(lbl) < MAX_TEXT_LENGTH:
                lbl = np.concatenate([
                    lbl,
                    np.zeros(MAX_TEXT_LENGTH - len(lbl), dtype=np.int32)
                ])
            else:
                lbl = lbl[:MAX_TEXT_LENGTH]
                actual_len = MAX_TEXT_LENGTH
 
            images.append(arr)
            labels.append(lbl)
            classes.append(cls)
 
        except Exception as e:
            print(f"  [SKIP] {Path(npy_path).name}: {e}")
            continue
 
    if not images:
        print(f"  [WARNING] Tidak ada data untuk {split}")
        return None, None, None
 
    return (
        np.array(images, dtype=np.float32),
        np.array(labels, dtype=np.int32),
        classes
    )

In [5]:
def ctc_loss_fn(y_true, y_pred):
    """
    Custom CTC Loss untuk training.
    
    ⚠️ PENTING: Hitung actual label length dari non-zero elements!
    Jangan gunakan padded length karena akan confuse CTC loss.
    """
    batch_len = tf.cast(tf.shape(y_true)[0], tf.int64)
    input_len = tf.cast(tf.shape(y_pred)[1], tf.int64) * \
                tf.ones(shape=(batch_len,), dtype=tf.int64)
    
    label_len = tf.reduce_sum(
        tf.cast(tf.not_equal(y_true, 0), tf.int64),
        axis=1
    )
    
    # Hitung CTC loss
    loss = tf.nn.ctc_loss(
        labels=tf.cast(y_true, tf.int32),
        logits=y_pred,
        label_length=label_len,
        logit_length=input_len,
        logits_time_major=False,
        blank_index=-1
    )
    return loss

In [6]:
class OCRCallback(keras.callbacks.Callback):
    """
    Custom callback untuk monitoring training OCR.
    - Print ringkasan tiap epoch
    - Early stopping jika val_loss tidak membaik
    - Simpan checkpoint terbaik
    - Logging ke file CSV
    """
    def __init__(self, patience=PATIENCE,
                 checkpoint_dir=CHECKPOINT_DIR,
                 log_path=os.path.join(PROJECT_ROOT, "training_log.csv")):
        super().__init__()
        self.patience         = patience
        self.checkpoint_dir   = checkpoint_dir
        self.log_path         = log_path
        self.best_loss        = np.inf
        self.wait             = 0
        self.logs_data        = []
 
    def on_epoch_end(self, epoch, logs=None):
        logs      = logs or {}
        val_loss  = logs.get("val_loss", np.inf)
        train_loss= logs.get("loss", 0)
 
        row = {
            "epoch": epoch + 1,
            "loss": train_loss,
            "val_loss": val_loss,
        }
        self.logs_data.append(row)
        pd.DataFrame(self.logs_data).to_csv(self.log_path, index=False)
 
        print(f"\n  [Epoch {epoch+1:03d}/{self.model.epochs}] "
              f"loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
              f"best={self.best_loss:.4f} | wait={self.wait}/{self.patience}")
 
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.wait      = 0
            
            # Simpan checkpoint
            ckpt_path = os.path.join(
                self.checkpoint_dir,
                f"best_epoch_{epoch+1:03d}_loss_{val_loss:.4f}.keras"
            )
            self.model.save(ckpt_path)
            print(f"    ✅ Val loss membaik → checkpoint disimpan")
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print(f"    🛑 Early stopping (wait {self.wait}/{self.patience})")
                self.model.stop_training = True
 
    def on_train_end(self, logs=None):
        print(f"\n[TRAINING SELESAI]")
        print(f"  Best val_loss : {self.best_loss:.4f}")
        print(f"  Log file      : {self.log_path}")

In [7]:
def compute_metrics(model, X, y_true_encoded, inference_model):
    """
    Hitung Sequence Accuracy dan Character Accuracy (1 - CER).
 
    Sequence Accuracy : prediksi teks == ground truth (exact match)
    Character Accuracy: 1 - CER (Character Error Rate)
      CER = edit_distance(pred, true) / len(true)
    """
    import sys
 
    # Forward pass pakai inference model (tanpa CTCLayer)
    y_pred = inference_model.predict(X, verbose=0)
 
    # Decode prediksi dengan CTC beam search
    input_len = np.ones(y_pred.shape[0], dtype=np.int32) * y_pred.shape[1]
    decoded, _ = tf.keras.backend.ctc_decode(
        y_pred,
        input_length=input_len,
        greedy=True
    )
    pred_indices = decoded[0].numpy()
 
    seq_correct = 0
    total_cer   = 0.0
 
    for i in range(len(X)):
        # Decode prediksi → teks
        pred_chars = []
        for idx in pred_indices[i]:
            if idx >= 0:
                pred_chars.append(idx_to_char.get(int(idx) + 1, ""))
        pred_text = "".join(pred_chars).strip()
 
        # Decode ground truth → teks
        true_chars = []
        for idx in y_true_encoded[i]:
            if idx > 0:
                true_chars.append(idx_to_char.get(int(idx), ""))
        true_text = "".join(true_chars).strip()
 
        # Sequence Accuracy — exact match
        if pred_text == true_text:
            seq_correct += 1
 
        # CER — edit distance / length ground truth
        if len(true_text) > 0:
            dist = edit_distance(pred_text, true_text)
            cer  = dist / len(true_text)
            total_cer += min(cer, 1.0)  # cap di 1.0
 
    n              = len(X)
    seq_acc        = seq_correct / n
    avg_cer        = total_cer / n
    char_acc       = 1.0 - avg_cer
 
    return seq_acc, char_acc
 
def edit_distance(s1: str, s2: str) -> int:
    """Hitung Levenshtein edit distance antara 2 string"""
    m, n = len(s1), len(s2)
    dp   = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp[m][n]
 
def build_inference_model_from(trained_model):
    """
    Buat inference model dari trained model dengan copy bobot.
    Inference model tidak punya CTCLayer — langsung output softmax.
    """
    from src.ocr_model import build_ocr_model as _build
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    from src.ocr_config import IMG_H, IMG_W, CHANNELS, NUM_CLASSES
 
    inp = keras.Input(shape=(IMG_H, IMG_W, CHANNELS), name="image")
    for layer in trained_model.layers:
        if layer.name == "image":
            x = inp
        elif layer.name in ("label", "ctc_loss"):
            continue
        else:
            try:
                x = layer(x)
            except Exception:
                continue
 
    inf_model = keras.Model(inputs=inp, outputs=x, name="OCR_inference")
 
    # Copy bobot dari trained model
    for layer in inf_model.layers:
        try:
            src = trained_model.get_layer(layer.name)
            layer.set_weights(src.get_weights())
        except Exception:
            continue
 
    return inf_model
 
# Mapping idx → char untuk decode
idx_to_char = {i + 1: c for i, c in enumerate(CHARACTERS)}
def train_with_gradient_tape(model, X_train, y_train,
                              X_valid, y_valid,
                              optimizer, epochs=EPOCHS,
                              batch_size=BATCH_SIZE):
    """
    Training loop manual dengan tf.GradientTape.
    Memberikan kontrol penuh atas gradient computation.
    """
    train_ds = tf.data.Dataset.from_tensor_slices(
        ({"image": X_train, "label": y_train}, y_train)
    ).shuffle(5000).batch(batch_size)
 
    valid_ds = tf.data.Dataset.from_tensor_slices(
        ({"image": X_valid, "label": y_valid}, y_valid)
    ).batch(batch_size)
 
    # TensorBoard writer
    writer = tf.summary.create_file_writer(LOG_DIR)
 
    # Build inference model untuk compute metrics
    inference_model = build_inference_model_from(model)
 
    best_val_loss = np.inf
    patience      = PATIENCE
    wait          = 0
    history       = []
 
    print(f"\n[TRAINING] tf.GradientTape loop")
    print(f"  Epochs          : {epochs}")
    print(f"  Batch size      : {batch_size}")
    print(f"  Train samples   : {len(X_train)}")
    print(f"  Valid samples   : {len(X_valid)}")
    print(f"  Learning rate   : {optimizer.learning_rate.numpy()}")
    print()
 
    for epoch in range(epochs):
        # ── Training ──────────────────────────────────────
        train_losses = []
        for batch_x, batch_y in train_ds:
            with tf.GradientTape() as tape:
                y_pred = model(batch_x, training=True)
                loss   = ctc_loss_fn(
                    tf.cast(batch_x["label"], tf.float32),
                    y_pred
                )
                loss   = tf.reduce_mean(loss)
 
            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(
                zip(grads, model.trainable_variables)
            )
            train_losses.append(float(loss))
 
        # ── Validation ────────────────────────────────────
        val_losses = []
        for batch_x, batch_y in valid_ds:
            y_pred   = model(batch_x, training=False)
            val_loss = ctc_loss_fn(
                tf.cast(batch_x["label"], tf.float32),
                y_pred
            )
            val_losses.append(float(tf.reduce_mean(val_loss)))
 
        avg_train = np.mean(train_losses)
        avg_val   = np.mean(val_losses)
 
        # ── Update inference model weights ────────────────
        for layer in inference_model.layers:
            try:
                src = model.get_layer(layer.name)
                layer.set_weights(src.get_weights())
            except Exception:
                continue
 
        # ── Hitung Sequence Accuracy + Character Accuracy ─
        # Pakai subset valid (max 200 sampel) agar tidak lama
        n_eval     = min(200, len(X_valid))
        seq_acc, char_acc = compute_metrics(
            model,
            X_valid[:n_eval],
            y_valid[:n_eval],
            inference_model
        )
 
        history.append({
            "epoch"    : epoch + 1,
            "loss"     : avg_train,
            "val_loss" : avg_val,
            "seq_acc"  : round(seq_acc  * 100, 2),
            "char_acc" : round(char_acc * 100, 2),
        })
 
        # TensorBoard logging
        with writer.as_default():
            tf.summary.scalar("loss",      avg_train,       step=epoch)
            tf.summary.scalar("val_loss",  avg_val,         step=epoch)
            tf.summary.scalar("seq_acc",   seq_acc * 100,   step=epoch)
            tf.summary.scalar("char_acc",  char_acc * 100,  step=epoch)
 
        print(f"  Epoch {epoch+1:03d}/{epochs} | "
              f"loss={avg_train:.4f} | val_loss={avg_val:.4f} | "
              f"seq_acc={seq_acc*100:.1f}% | char_acc={char_acc*100:.1f}% | "
              f"wait={wait}/{patience}")
 
        # Early stopping
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            wait          = 0
 
            # Simpan best model
            best_path = os.path.join(CHECKPOINT_DIR, "best_model.keras")
            model.save(best_path)
            print(f"    ✅ Val loss membaik → best model disimpan")
        else:
            wait += 1
            if wait >= patience:
                print(f"    🛑 Early stopping")
                break
 
    return history

In [8]:
print("[DATA] Memuat dataset...")
X_train, y_train, cls_train = load_dataset_from_npy_and_csv("train")
X_valid, y_valid, cls_valid = load_dataset_from_npy_and_csv("valid")
X_test,  y_test,  cls_test  = load_dataset_from_npy_and_csv("test")
 
print()
for split, X in [("train", X_train),
                  ("valid", X_valid),
                  ("test",  X_test)]:
    if X is not None:
        print(f"  {split:5s}: {len(X):4d} sampel → shape={X.shape}")
    else:
        print(f"  {split:5s}: tidak ada data ❌")
 
# Build model
print(f"\n[MODEL] Membangun model OCR...")
model = build_ocr_model()
print(f"  Total params: {model.count_params():,}")
 
if X_train is not None and X_valid is not None:
    # Setup training
    optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE)
 
    print(f"\n[TRAINING] Mulai training dengan tf.GradientTape...")
    
    history = train_with_gradient_tape(
        model, X_train, y_train,
        X_valid, y_valid,
        optimizer=optimizer,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE
    )
 
    # Simpan history
    pd.DataFrame(history).to_csv(
        os.path.join(PROJECT_ROOT, "training_history.csv"),
        index=False
    )
 
    print(f"\n[INFO] Model checkpoints tersimpan di:")
    print(f"       {CHECKPOINT_DIR}")
    print(f"       ")
    print(f"  ⏸️  Export ke .keras / SavedModel menunggu:")
    print(f"      - Teman selesai evaluasi performa")
    print(f"      - Approval untuk final model")
 
    # Simpan config training (untuk reference)
    config = {
        "model_name": "MoneyLens_OCR",
        "architecture": "CNN (32-64-128) + BiLSTM(128-64) + CTC",
        "input_shape": (IMG_H, IMG_W, CHANNELS),
        "num_classes": NUM_CLASSES,
        "training": {
            "epochs_trained": len(history),
            "best_val_loss": float(min(h["val_loss"] for h in history)),
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
        },
        "data": {
            "train_samples": len(X_train) if X_train is not None else 0,
            "valid_samples": len(X_valid) if X_valid is not None else 0,
            "test_samples": len(X_test) if X_test is not None else 0,
        },
        "status": "Training selesai - menunggu evaluasi teman sebelum final export"
    }
    config_path = os.path.join(PROJECT_ROOT, "training_config.json")
    with open(config_path, "w") as f:
        json.dump(config, f, indent=2)
    print(f"  ✅ Config       : {config_path}")
 
else:
    print("\n[ERROR] Data tidak tersedia!")
    print("        Pastikan Task 2 preprocessing sudah selesai.")
 
print(f"\n{'='*70}")
print("TRAINING SELESAI")
print(f"{'='*70}")
print(f"  Model checkpoints : {CHECKPOINT_DIR}")
print(f"  Training history  : {os.path.join(PROJECT_ROOT, 'training_history.csv')}")
print(f"  TensorBoard       : tensorboard --logdir={LOG_DIR}")
print(f"  Training log      : {os.path.join(PROJECT_ROOT, 'training_log.csv')}")
print(f"  ")
print(f"  ⏸️  Export ke .keras / SavedModel:")
print(f"      Tunggu teman selesai evaluasi performa")
print(f"{'='*70}")

[DATA] Memuat dataset...
  train: 3194 anotasi | kelas: ['QTY', 'harga_satuan', 'nama_produk', 'tanggal', 'total_harga_barang', 'total_transaksi']
  3194 file .npy ditemukan
  valid: 915 anotasi | kelas: ['QTY', 'harga_satuan', 'nama_produk', 'tanggal', 'total_harga_barang', 'total_transaksi']
  915 file .npy ditemukan
  test: 422 anotasi | kelas: ['QTY', 'harga_satuan', 'nama_produk', 'tanggal', 'total_harga_barang', 'total_transaksi']
  422 file .npy ditemukan

  train: 3194 sampel → shape=(3194, 32, 128, 1)
  valid:  915 sampel → shape=(915, 32, 128, 1)
  test :  422 sampel → shape=(422, 32, 128, 1)

[MODEL] Membangun model OCR...
  Total params: 497,672

[TRAINING] Mulai training dengan tf.GradientTape...

[TRAINING] tf.GradientTape loop
  Epochs          : 100
  Batch size      : 32
  Train samples   : 3194
  Valid samples   : 915
  Learning rate   : 0.0010000000474974513



I0000 00:00:1779313113.305652  408704 cuda_dnn.cc:461] Loaded cuDNN version 92200


  Epoch 001/100 | loss=31.7867 | val_loss=27.0203 | seq_acc=0.0% | char_acc=0.0% | wait=0/10
    ✅ Val loss membaik → best model disimpan
  Epoch 002/100 | loss=23.0849 | val_loss=30.4824 | seq_acc=0.0% | char_acc=0.0% | wait=0/10
  Epoch 003/100 | loss=18.8613 | val_loss=33.2412 | seq_acc=0.0% | char_acc=0.0% | wait=1/10
  Epoch 004/100 | loss=16.2842 | val_loss=37.4895 | seq_acc=0.0% | char_acc=1.4% | wait=2/10
  Epoch 005/100 | loss=14.3952 | val_loss=30.5853 | seq_acc=0.0% | char_acc=1.4% | wait=3/10
  Epoch 006/100 | loss=13.2422 | val_loss=20.4390 | seq_acc=0.0% | char_acc=0.2% | wait=4/10
    ✅ Val loss membaik → best model disimpan
  Epoch 007/100 | loss=12.3852 | val_loss=23.5979 | seq_acc=0.0% | char_acc=0.2% | wait=0/10
  Epoch 008/100 | loss=11.8307 | val_loss=15.9076 | seq_acc=0.0% | char_acc=2.0% | wait=1/10
    ✅ Val loss membaik → best model disimpan
  Epoch 009/100 | loss=11.3534 | val_loss=13.4333 | seq_acc=0.0% | char_acc=2.0% | wait=0/10
    ✅ Val loss membaik → bes

In [9]:
import pandas as pd

log_path = r"/home/hasan/coding/MoneyLens/ai/training_history.csv"
df = pd.read_csv(log_path)

print("Training History:")
print(df.to_string())
print(f"\nBest val_loss : {df['val_loss'].min():.4f} (epoch {df['val_loss'].idxmin()+1})")
print(f"Final loss    : {df['loss'].iloc[-1]:.4f}")
print(f"Total epochs  : {len(df)}")

Training History:
    epoch       loss   val_loss  seq_acc  char_acc
0       1  31.786720  27.020319      0.0      0.00
1       2  23.084923  30.482365      0.0      0.00
2       3  18.861322  33.241170      0.0      0.00
3       4  16.284171  37.489459      0.0      1.43
4       5  14.395195  30.585255      0.0      1.36
5       6  13.242206  20.439034      0.0      0.22
6       7  12.385182  23.597928      0.0      0.17
7       8  11.830701  15.907604      0.0      2.00
8       9  11.353412  13.433314      0.0      1.98
9      10  11.072257  25.044421      0.0      0.73
10     11  10.730541  13.250536      0.0      2.07
11     12  10.469771  17.960087      0.0      0.46
12     13  10.367511  13.866201      0.0      1.03
13     14  10.006663  36.435791      0.0      1.89
14     15   9.838558  13.215739      0.0      3.14
15     16   9.645191  13.737385      0.0      2.34
16     17   9.509295  12.996497      0.0      1.72
17     18   9.572937  26.345562      0.0      0.95
18     19   9